# Compare Two Allocation Strategies

Compare **VIX-based rebalance** (QQQ/BIL/GLD with high/low vol bands) vs **long-hold QQQ** over the same period. Each engine receives symbols and date range and fetches data itself. Set `ALPACA_API_KEY` and `ALPACA_API_SECRET` (or rely on Yahoo Finance fallback) and optional cache.

In [1]:
from tiportfolio.helpers.cache import enable_data_source_cache
from tiportfolio import (
    FixRatio,
    Schedule,
    ScheduleBasedEngine,
    VixRegimeAllocation,
    VolatilityBasedEngine,
    compare_strategies,
)
import pandas as pd

from dotenv import load_dotenv
load_dotenv()

enable_data_source_cache("tiportfolio", cache_dir=".cache")

SYMBOLS = ["QQQ", "BIL", "GLD"]
VOLATILITY_SYMBOL = "VIX"
START = "2018-01-01"
END = "2024-12-31"
INITIAL_VALUE = 10_000
FEE_PER_SHARE = 0.0035

## VIX Target Rebalance

In [2]:
## VIX Target Rebalance

# Strategy 1: VIX-based rebalance — high-vol (0.4, 0.4, 0.2), low-vol (0.7, 0.2, 0.1), target 15, bounds -1 / 5
high_weights = {"QQQ": 0.4, "BIL": 0.4, "GLD": 0.2}
low_weights = {"QQQ": 0.7, "BIL": 0.2, "GLD": 0.1}

# API CHANGE NOTE: After the volatility engine refactor:
# - OLD: VixRegimeAllocation(target_vix=X, lower_bound=Y, upper_bound=Z, ...)
# - NEW: VixRegimeAllocation(high_vol_allocation=..., low_vol_allocation=...)
# - VIX parameters (target_vix, lower_bound, upper_bound) are now passed to VolatilityBasedEngine.run()
# This change separates concerns: Allocation strategies decide weights, Engine decides when to use which allocation
allocation_vix = VixRegimeAllocation(
    high_vol_allocation=FixRatio(weights=high_weights),
    low_vol_allocation=FixRatio(weights=low_weights),
)
engine1 = VolatilityBasedEngine(
    allocation=allocation_vix,
    rebalance=Schedule("vix_regime"),
    fee_per_share=FEE_PER_SHARE,
    initial_value=INITIAL_VALUE,
)

# Load VIX data from CSV
vix_df = pd.read_csv('../tests/data/vix_2018_2024_df.csv', index_col='date', parse_dates=True)
# Add OHLC columns if missing
if 'open' not in vix_df.columns:
    vix_df['open'] = vix_df['close']
    vix_df['high'] = vix_df['close']
    vix_df['low'] = vix_df['close']
# Normalize index
from tiportfolio.calendar import normalize_price_index
vix_df = normalize_price_index(vix_df)

result_vix_regime = engine1.run(
    symbols=SYMBOLS,
    start=START,
    end=END,
    volatility_symbol=VOLATILITY_SYMBOL,
    vix_df=vix_df,
    
    target_vix=15.0,    # These VIX parameters moved here from constructor
    lower_bound=-1.0,    # These VIX parameters moved here from constructor  
    upper_bound=5.0,      # These VIX parameters moved here from constructor
)

Loaded cached bar data.



In [3]:
print(result_vix_regime.summary())

Backtest Summary
----------------
Sharpe Ratio:        0.6618
Sortino Ratio:       0.8696
MAR Ratio:           0.6698
CAGR:                11.57%
Max Drawdown:        17.27%
Kelly Leverage:      5.7366
Mean Excess Return:  0.0764
Final Value:         21,499.97
Total Fee:           3.97
Rebalances:          113


In [4]:
from tiportfolio.report import rebalance_decisions_table
rebalance_decisions_table(result_vix_regime)

,date,equity_before,equity_after,fee_paid,QQQ_price,QQQ_qty_before,QQQ_trade_qty,QQQ_qty_after,QQQ_value_after,BIL_price,BIL_qty_before,BIL_trade_qty,BIL_qty_after,BIL_value_after,GLD_price,GLD_qty_before,GLD_trade_qty,GLD_qty_after,GLD_value_after
0,2018-02-01 00:00:00-05:00,10444.058,10444.051,0.007,158.89,46.688,-0.676,46.012,7310.841,75.17,26.638,1.150,27.788,2088.812,128.07,7.990,0.165,8.155,1044.406
1,2018-02-06 00:00:00-05:00,10176.695,10176.506,0.189,153.55,46.012,-19.502,26.510,4070.678,75.18,27.788,26.358,54.146,4070.678,125.38,8.155,8.078,16.233,2035.339
2,2018-02-21 00:00:00-05:00,10245.153,10245.151,0.002,155.92,26.510,-0.227,26.283,4098.061,75.20,54.146,0.350,54.495,4098.061,125.66,16.233,0.073,16.306,2049.031
3,2018-03-02 00:00:00-05:00,10272.323,10272.322,0.001,157.03,26.283,-0.117,26.167,4108.929,75.25,54.495,0.108,54.604,4108.929,125.37,16.306,0.081,16.387,2054.465
4,2018-03-23 00:00:00-04:00,10132.880,10132.872,0.007,150.20,26.167,0.819,26.985,4053.152,75.30,54.604,-0.777,53.827,4053.152,127.60,16.387,-0.505,15.882,2026.576
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108,2024-10-28 00:00:00-04:00,21264.574,21264.570,0.004,492.14,17.282,0.001,17.283,8505.829,86.76,97.278,0.760,98.039,8505.829,253.33,17.051,-0.263,16.788,4252.915
109,2024-10-31 00:00:00-04:00,21074.257,21074.253,0.005,480.67,17.283,0.254,17.537,8429.703,86.81,98.039,-0.933,97.105,8429.703,253.51,16.788,-0.162,16.626,4214.851
110,2024-12-02 00:00:00-05:00,21487.543,21487.307,0.236,511.90,17.537,11.846,29.383,15041.280,87.15,97.105,-47.794,49.312,4297.509,243.44,16.626,-7.799,8.827,2148.754
111,2024-12-12 00:00:00-05:00,21855.177,21855.174,0.003,523.04,29.383,-0.134,29.249,15298.624,87.28,49.312,0.769,50.081,4371.035,247.28,8.827,0.012,8.838,2185.518


In [5]:

fig = result_vix_regime.plot_portfolio(mark_rebalances=True)
fig.show()

In [6]:
from tiportfolio.report import rebalance_decisions_table, plot_strategy_comparison_interactive

decisions_df = rebalance_decisions_table(result_vix_regime)
decisions_df.head(10)

,date,equity_before,equity_after,fee_paid,QQQ_price,QQQ_qty_before,QQQ_trade_qty,QQQ_qty_after,QQQ_value_after,BIL_price,BIL_qty_before,BIL_trade_qty,BIL_qty_after,BIL_value_after,GLD_price,GLD_qty_before,GLD_trade_qty,GLD_qty_after,GLD_value_after
0,2018-02-01 00:00:00-05:00,10444.058,10444.051,0.007,158.89,46.688,-0.676,46.012,7310.841,75.17,26.638,1.150,27.788,2088.812,128.07,7.990,0.165,8.155,1044.406
1,2018-02-06 00:00:00-05:00,10176.695,10176.506,0.189,153.55,46.012,-19.502,26.510,4070.678,75.18,27.788,26.358,54.146,4070.678,125.38,8.155,8.078,16.233,2035.339
2,2018-02-21 00:00:00-05:00,10245.153,10245.151,0.002,155.92,26.510,-0.227,26.283,4098.061,75.20,54.146,0.350,54.495,4098.061,125.66,16.233,0.073,16.306,2049.031
3,2018-03-02 00:00:00-05:00,10272.323,10272.322,0.001,157.03,26.283,-0.117,26.167,4108.929,75.25,54.495,0.108,54.604,4108.929,125.37,16.306,0.081,16.387,2054.465
4,2018-03-23 00:00:00-04:00,10132.880,10132.872,0.007,150.20,26.167,0.819,26.985,4053.152,75.30,54.604,-0.777,53.827,4053.152,127.60,16.387,-0.505,15.882,2026.576
5,2018-04-03 00:00:00-04:00,10081.467,10081.465,0.002,149.02,26.985,0.076,27.061,4032.587,75.32,53.827,-0.287,53.539,4032.587,126.30,15.882,0.082,15.964,2016.293
6,2018-04-09 00:00:00-04:00,10103.281,10103.281,0.001,149.46,27.061,-0.021,27.039,4041.313,75.35,53.539,0.094,53.634,4041.313,126.82,15.964,-0.031,15.933,2020.656
7,2018-05-10 00:00:00-04:00,10387.248,10387.067,0.182,160.73,27.039,18.198,45.238,7271.074,75.45,53.634,-26.100,27.534,2077.450,125.18,15.933,-7.635,8.298,1038.725
8,2018-05-17 00:00:00-04:00,10309.209,10309.208,0.001,159.51,45.238,0.004,45.241,7216.446,75.47,27.534,-0.214,27.320,2061.842,122.36,8.298,0.127,8.425,1030.921
9,2018-06-04 00:00:00-04:00,10566.988,10566.983,0.005,165.16,45.241,-0.455,44.786,7396.891,75.54,27.320,0.657,27.977,2113.398,122.39,8.425,0.209,8.634,1056.699


## Long Hold QQQ

In [7]:
# Strategy 2: Long hold QQQ (no rebalance)
engine2 = ScheduleBasedEngine(
    allocation=FixRatio(weights={"QQQ": 1.0}),
    rebalance=Schedule("month_start"),
    fee_per_share=FEE_PER_SHARE,
    initial_value=INITIAL_VALUE,
)
result_qqq_only = engine2.run(
    symbols=["QQQ"],
    start=START,
    end=END,
)
print(result_qqq_only.summary())

Loaded cached bar data.

Backtest Summary
----------------
Sharpe Ratio:        0.6860
Sortino Ratio:       0.8910
MAR Ratio:           0.5495
CAGR:                19.24%
Max Drawdown:        35.01%
Kelly Leverage:      2.8433
Mean Excess Return:  0.1655
Final Value:         34,218.64
Total Fee:           0.00
Rebalances:          0


In [8]:
compare_strategies(
    result_vix_regime,
    result_qqq_only,
    names=['vix', 'long_qqq']
)

,vix,long_qqq,better
metric,,,
sharpe_ratio,0.661822,0.685981,long_qqq
sortino_ratio,0.869554,0.891008,long_qqq
mar_ratio,0.669814,0.549477,vix
cagr,0.115687,0.192355,long_qqq
max_drawdown,0.172715,0.350069,vix


In [9]:
# Interactive comparison chart
plot_strategy_comparison_interactive(
    result_vix_regime,
    result_qqq_only,
    names=[
        "VIX regime (QQQ/BIL/GLD)",
        "Long QQQ",
    ]
)

## Schedule

In [10]:
schedule_engine = ScheduleBasedEngine(
    allocation=FixRatio(weights={"QQQ": 0.7, "BIL": 0.2, "GLD": 0.1}),
    rebalance=Schedule("month_start"),
    fee_per_share=0.0035,
    initial_value=INITIAL_VALUE,
)
schedule_result = schedule_engine.run(symbols=SYMBOLS, start=START, end=END,)
print(schedule_result.summary())

Loaded cached bar data.

Backtest Summary
----------------
Sharpe Ratio:        0.6911
Sortino Ratio:       0.8997
MAR Ratio:           0.5824
CAGR:                15.33%
Max Drawdown:        26.31%
Kelly Leverage:      4.0709
Mean Excess Return:  0.1173
Final Value:         27,102.06
Total Fee:           0.76
Rebalances:          83


## Use Freezing Period

In [11]:
## VIX Target Rebalance with Freezing (30 days)

engine_freezing = VolatilityBasedEngine(
    allocation=allocation_vix,
    rebalance=Schedule("vix_regime"),
    fee_per_share=FEE_PER_SHARE,
    initial_value=INITIAL_VALUE,
    freezing_days=30,
)
result_vix_freezing = engine_freezing.run(
    symbols=SYMBOLS,
    start=START,
    end=END,
    volatility_symbol=VOLATILITY_SYMBOL,
    target_vix=15.0,
    lower_bound=-1.0,
    upper_bound=5.0,
)
print(result_vix_freezing.summary())

Loaded cached bar data.

Loaded cached bar data.

Backtest Summary
----------------
Sharpe Ratio:        0.7071
Sortino Ratio:       0.8754
MAR Ratio:           0.6525
CAGR:                13.65%
Max Drawdown:        20.92%
Kelly Leverage:      5.1116
Mean Excess Return:  0.0978
Final Value:         24,465.96
Total Fee:           3.03
Rebalances:          44


In [12]:
compare_strategies(
    result_vix_regime,
    result_qqq_only,
    result_vix_freezing,
    names=["VIX Original",
           "qqq_only",
           "VIX with 30-day Freezing"])

,VIX Original,qqq_only,VIX with 30-day Freezing,better
metric,,,,
sharpe_ratio,0.661822,0.685981,0.707074,VIX with 30-day Freezing
sortino_ratio,0.869554,0.891008,0.875423,qqq_only
mar_ratio,0.669814,0.549477,0.652532,VIX Original
cagr,0.115687,0.192355,0.136498,qqq_only
max_drawdown,0.172715,0.350069,0.209182,VIX Original


In [13]:
# Interactive comparison chart
plot_strategy_comparison_interactive(
    result_vix_regime,
    result_qqq_only,
    result_vix_freezing,
    names=[
        "VIX regime (QQQ/BIL/GLD)",
        "Long QQQ",
        "Result freezing",
    ]
)

## Comparing All Four

In [14]:
compare_strategies(
    result_vix_regime,
    result_qqq_only,
    schedule_result,
    result_vix_freezing
)

,Strategy 1,Strategy 2,Strategy 3,Strategy 4,better
metric,,,,,
sharpe_ratio,0.661822,0.685981,0.691077,0.707074,Strategy 4
sortino_ratio,0.869554,0.891008,0.899677,0.875423,Strategy 3
mar_ratio,0.669814,0.549477,0.582384,0.652532,Strategy 1
cagr,0.115687,0.192355,0.153252,0.136498,Strategy 2
max_drawdown,0.172715,0.350069,0.263145,0.209182,Strategy 1


In [15]:
plot_strategy_comparison_interactive(
    result_vix_regime,
    result_qqq_only,
    schedule_result,
    result_vix_freezing,)

### Use Leverage

In [16]:
compare_strategies(
    result_vix_regime,
    result_qqq_only,
    schedule_result,
    result_vix_freezing,
    leverages=[
        2.8,
        1,
        1.9,
        1.9,
    ],
     yearly_loan_rates=0.05,
)

,"Strategy 1 (L2.8x, r5.0%)",Strategy 2,"Strategy 3 (L1.9x, r5.0%)","Strategy 4 (L1.9x, r5.0%)",better
metric,,,,,
sharpe_ratio,0.661822,0.685981,0.691077,0.707074,"Strategy 4 (L1.9x, r5.0%)"
sortino_ratio,0.869554,0.891008,0.899677,0.875423,"Strategy 3 (L1.9x, r5.0%)"
mar_ratio,0.483710,0.549477,0.492380,0.539309,Strategy 2
cagr,0.233923,0.192355,0.246178,0.214346,"Strategy 3 (L1.9x, r5.0%)"
max_drawdown,0.483601,0.350069,0.499976,0.397446,Strategy 2


In [17]:
plot_strategy_comparison_interactive(
    result_vix_regime,
    result_qqq_only,
    schedule_result,
    result_vix_freezing,
    leverages=[
        2.8,
        1,
        1.9,
        1.9,
    ],
     yearly_loan_rates=0.05,
)

## Portfolio Beta

Track portfolio beta over time vs SPY.


In [18]:
# Plot portfolio beta over time
fig_beta = result_vix_regime.plot_portfolio_beta()
fig_beta.show()


Loaded cached bar data.

